## ***Initials***

In [2]:
import pandas as pd
import numpy as np
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


## ***Approach 1 - Features of just Neighborhoods***

### ***Load the Data***

In [3]:
# === 1. Load the data ===
df = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\4. Joining Datasets\\3. Joining - Business - Neighborhood Data\\Extracted CSV Files\\business_data_neighborhood.csv") 
print(df.shape)
print(df.columns)

(3882, 22)
Index(['Name', 'Category', 'Latitude', 'Longitude', 'Address', 'Country',
       'City', 'Postal Code', 'Rating', 'Reviews', 'Source', 'Description',
       'NACE Code', 'NACE Description (EN)', 'Municipal_Community',
       'Neighborhood', 'Neighborhood_Area_km2', 'distance_to_volos_center_km',
       'distance_to_volos_port_km', 'dist_to_main_road_km',
       'dist_to_bus_stop_km', 'dist_to_university_km'],
      dtype='object')


### ***Feature Selection***

In [4]:
# The seleceted features are chosen based on the EDA performed earlier
selected_features = [

    # Business-level features
    'Latitude', 'Longitude', 'NACE Code',
    
    # Neighborhood-level features
    'Neighborhood_Area_km2', 'distance_to_volos_center_km',
       'distance_to_volos_port_km', 'dist_to_main_road_km',
       'dist_to_bus_stop_km', 'dist_to_university_km',

    # Rating
    'Rating',  # This is the target
]

In [5]:
df = df[selected_features].copy()

print(df.shape)
print(df.columns)
df.head()

(3882, 10)
Index(['Latitude', 'Longitude', 'NACE Code', 'Neighborhood_Area_km2',
       'distance_to_volos_center_km', 'distance_to_volos_port_km',
       'dist_to_main_road_km', 'dist_to_bus_stop_km', 'dist_to_university_km',
       'Rating'],
      dtype='object')


,Latitude,Longitude,NACE Code,Neighborhood_Area_km2,distance_to_volos_center_km,distance_to_volos_port_km,dist_to_main_road_km,dist_to_bus_stop_km,dist_to_university_km,Rating
0,39.335293,22.923506,47.76,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331,NaN
1,39.339233,22.923969,47.59,4.848904,3.508166,2.529372,0.224486,0.246542,1.913456,NaN
2,39.332305,22.926496,94.91,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331,NaN
3,39.332756,22.929457,49.41,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331,NaN
4,39.332876,22.929374,47.11,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331,NaN


### ***Divide Labeled vs Unlabeled Entries***

In [6]:
# === 2. Preprocessing ===

# Separate labeled vs unlabeled
df_labeled = df[df['Rating'].notna()].copy()
df_unlabeled = df[df['Rating'].isna()].copy()

print(f"Number of labeled samples: {len(df_labeled)}")
print(f"Number of unlabeled samples: {len(df_unlabeled)}")

Number of labeled samples: 228
Number of unlabeled samples: 3654


In [7]:
X_labeled = df_labeled.drop(columns='Rating')
y_labeled = df_labeled['Rating']
X_unlabeled = df_unlabeled.drop(columns='Rating')

print(f"{len(X_labeled.columns)} colunms in X_labeled: {X_labeled.columns}\n")
print(f"Target column: y_labeled, dtype: {y_labeled.dtype}\n")
print(f"{len(X_unlabeled.columns)} colunms in X_unlabeled: {X_unlabeled.columns}\n")

9 colunms in X_labeled: Index(['Latitude', 'Longitude', 'NACE Code', 'Neighborhood_Area_km2',
       'distance_to_volos_center_km', 'distance_to_volos_port_km',
       'dist_to_main_road_km', 'dist_to_bus_stop_km', 'dist_to_university_km'],
      dtype='object')

Target column: y_labeled, dtype: float64

9 colunms in X_unlabeled: Index(['Latitude', 'Longitude', 'NACE Code', 'Neighborhood_Area_km2',
       'distance_to_volos_center_km', 'distance_to_volos_port_km',
       'dist_to_main_road_km', 'dist_to_bus_stop_km', 'dist_to_university_km'],
      dtype='object')



### ***Encode NACE Code as Categorical Feature***

In [8]:
# Identify numerical and categorical columns
categorical_cols = ['NACE Code']
numerical_cols = [col for col in selected_features if col != 'NACE Code' and col != 'Rating']
print(f"{len(categorical_cols)} categorical columns: {categorical_cols}")
print(f"{len(numerical_cols)} numerical columns: {numerical_cols}")

1 categorical columns: ['NACE Code']
8 numerical columns: ['Latitude', 'Longitude', 'Neighborhood_Area_km2', 'distance_to_volos_center_km', 'distance_to_volos_port_km', 'dist_to_main_road_km', 'dist_to_bus_stop_km', 'dist_to_university_km']


In [9]:
# Preprocess data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ]), numerical_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ])

In [10]:
# Apply preprocessing
X_labeled_processed = preprocessor.fit_transform(X_labeled)
X_unlabeled_processed = preprocessor.transform(X_unlabeled)

### ***Perform Self-Training***

In [11]:
# Initialize regressor
base_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

In [12]:
# Self-training loop
confidence_threshold = 0.1  # threshold for high-confidence predictions
max_iterations = 10

X_train = X_labeled_processed
y_train = y_labeled
X_unlabeled_pool = X_unlabeled_processed.copy()


for iteration in range(max_iterations):

    # Train the model
    base_regressor.fit(X_train, y_train)

    # Predict the unlabeled data target
    predictions = base_regressor.predict(X_unlabeled_pool)

    # Find the confident predictions
    tree_predictions = np.array([tree.predict(X_unlabeled_pool) for tree in base_regressor.estimators_])
    variances = np.var(tree_predictions, axis=0)
    high_confidence_idx = variances < confidence_threshold
    
    if not high_confidence_idx.any():
        print(f"No high-confidence predictions in iteration {iteration + 1}. Stopping.")
        break


    # Add the confident predictions to the labeled set
    X_new_labeled = X_unlabeled_pool[high_confidence_idx]
    y_new_labeled = predictions[high_confidence_idx]
    
    # Update the training set
    X_train = np.vstack((X_train, X_new_labeled))
    y_train = np.hstack((y_train, y_new_labeled))
    
    # Remove the confident predictions from the unlabeled set
    X_unlabeled_pool = X_unlabeled_pool[~high_confidence_idx]
    print(f"Iteration {iteration + 1}: Added {len(X_new_labeled)} new labeled samples")
    
    if len(X_unlabeled_pool) == 0:
        print("All unlabeled data labeled.")
        break
    

Iteration 1: Added 1105 new labeled samples
Iteration 2: Added 608 new labeled samples
Iteration 3: Added 472 new labeled samples
Iteration 4: Added 80 new labeled samples
Iteration 5: Added 34 new labeled samples
Iteration 6: Added 12 new labeled samples
Iteration 7: Added 2 new labeled samples
Iteration 8: Added 2 new labeled samples
No high-confidence predictions in iteration 9. Stopping.


In [13]:
# Infrerence on the unlabeled data
final_predictions = base_regressor.predict(X_unlabeled_processed)

# Clip predictions to ensure they are within 1–5
final_predictions = np.clip(final_predictions, 1, 5)

# Update the original dataset
df.loc[df['Rating'].isnull(), 'Rating'] = final_predictions

In [14]:
# Assign the new ratings to the original DataFrame

original_df = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\4. Joining Datasets\\3. Joining - Business - Neighborhood Data\\Extracted CSV Files\\business_data_neighborhood.csv")

columns_to_compare = [col for col in df.columns if col != 'Rating']
matching_rows = (original_df[columns_to_compare] == df[columns_to_compare]).all(axis=1)
if not matching_rows.all():
    print("⚠️ Warning: Some rows do not match between the original and new DataFrames.")
    mismatches = (~matching_rows).sum()
    print(f"Non-matching rows: {mismatches}")


# Only update where original Rating is missing AND rows match
original_df.loc[original_df['Rating'].isna() & matching_rows, 'Rating'] = \
    df.loc[original_df['Rating'].isna() & matching_rows, 'Rating']

### ***Save the Data***

In [15]:
print(original_df.shape)
print(original_df.columns)
print(f"Missing ratings: {original_df['Rating'].isna().sum()}")
original_df.head()

(3882, 22)
Index(['Name', 'Category', 'Latitude', 'Longitude', 'Address', 'Country',
       'City', 'Postal Code', 'Rating', 'Reviews', 'Source', 'Description',
       'NACE Code', 'NACE Description (EN)', 'Municipal_Community',
       'Neighborhood', 'Neighborhood_Area_km2', 'distance_to_volos_center_km',
       'distance_to_volos_port_km', 'dist_to_main_road_km',
       'dist_to_bus_stop_km', 'dist_to_university_km'],
      dtype='object')
Missing ratings: 0


,Name,Category,Latitude,Longitude,Address,Country,City,Postal Code,Rating,Reviews,...,NACE Code,NACE Description (EN),Municipal_Community,Neighborhood,Neighborhood_Area_km2,distance_to_volos_center_km,distance_to_volos_port_km,dist_to_main_road_km,dist_to_bus_stop_km,dist_to_university_km
0,Άνθη-φυτά,Flower Store,39.335293,22.923506,6χλ.βολου,Greece,Βόλος Μαγνησίας,38334,2.754576,NaN,...,47.76,"Retail sale of flowers, plants, seeds, fertili...",Municipal Community of Volos,Nees Pagases,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331
1,Μέταλλο και ξύλο,Furniture and Home Store,39.339233,22.923969,Βόλου - Αθηνών 7ο χλμ,Greece,Βόλος Μαγνησίας,38334,4.932753,NaN,...,47.59,"Retail sale of furniture, lighting equipment a...",Municipal Community of Volos,Aivaliotika,4.848904,3.508166,2.529372,0.224486,0.246542,1.913456
2,Προφήτης Ηλίας Αλυκών,Church,39.332305,22.926496,Λεωφόρος Αθηνών 155,Greece,Βόλος Μαγνησίας,38334,2.754621,NaN,...,94.91,Activities of religious organisations,Municipal Community of Volos,Nees Pagases,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331
3,Frago Cargo (Φραγγοσ Νικολαοσ),"Shipping, Freight, and Material Transportation...",39.332756,22.929457,Αλόης 179Γ,Greece,Βόλος Μαγνησίας,38334,2.783596,NaN,...,49.41,Freight transport by road,Municipal Community of Volos,Nees Pagases,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331
4,Αλφα Ωμεγα Express Market,Grocery Store,39.332876,22.929374,Βάκχου 4,Greece,Βόλος Μαγνησίας,38334,3.142287,NaN,...,47.11,Retail sale in non-specialised stores with foo...,Municipal Community of Volos,Nees Pagases,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331


In [17]:
original_df.to_csv('C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\6. Ground Truths\\3. Semi-Supervised Approach\\1. Approach 1 (won)\\Extracted CSV Files\\business_data_imputed.csv', index=False)

## ***Approach 2 - Features of Neighborhoods AND Municipal Communities***

### ***Load the Data***

In [18]:
# === 1. Load the data ===
df = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\4. Joining Datasets\\3. Joining - Business - Neighborhood Data\\Extracted CSV Files\\business_data_ELSTAT_neighborhood.csv") 
print(df.shape)
print(df.columns)

(3882, 42)
Index(['Name', 'Category', 'Latitude', 'Longitude', 'Address', 'Country',
       'City', 'Postal Code', 'Rating', 'Reviews', 'Source', 'Description',
       'NACE Code', 'NACE Description (EN)', 'Municipal_Community',
       'Neighborhood', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ', 'ΔΗΜΟΣ', 'ΔΗΜΟΤΙΚΗ ΕΝΟΤΗΤΑ',
       'Population', 'Άγαμοι_pct', 'Έγγαμοι_pct', 'Χήροι_pct',
       'Διαζευγμένοι_pct', 'age_0_14_pct', 'age_15_64_pct', 'age_65_plus_pct',
       'low_education_pct', 'medium_education_pct', 'high_education_pct',
       'unemployment_rate', 'labor_force_participation_rate',
       'primary_sector_pct', 'secondary_sector_pct', 'tertiary_sector_pct',
       'Area_km2', 'Neighborhood_Area_km2', 'distance_to_volos_center_km',
       'distance_to_volos_port_km', 'dist_to_main_road_km',
       'dist_to_bus_stop_km', 'dist_to_university_km'],
      dtype='object')


### ***Feature Selection***

In [19]:
# The seleceted features are chosen based on the EDA performed earlier
selected_features = [

    # Business-level features
    'Latitude', 'Longitude', 'NACE Code',

    # Neighborhood-level features
    'Neighborhood_Area_km2', 'distance_to_volos_center_km',
       'distance_to_volos_port_km', 'dist_to_main_road_km',
       'dist_to_bus_stop_km', 'dist_to_university_km',

    # Municipal-Community-level features
    'Population', 'Άγαμοι_pct', 'Έγγαμοι_pct', 'Χήροι_pct',
       'Διαζευγμένοι_pct', 'age_0_14_pct', 'age_15_64_pct', 'age_65_plus_pct',
       'low_education_pct', 'medium_education_pct', 'high_education_pct',
       'unemployment_rate', 'labor_force_participation_rate',
       'primary_sector_pct', 'secondary_sector_pct', 'tertiary_sector_pct',
       'Area_km2',

    # Rating
    'Rating',  # This is the target
]

In [20]:
df = df[selected_features].copy()

print(df.shape)
print(df.columns)
df.head()

(3882, 27)
Index(['Latitude', 'Longitude', 'NACE Code', 'Neighborhood_Area_km2',
       'distance_to_volos_center_km', 'distance_to_volos_port_km',
       'dist_to_main_road_km', 'dist_to_bus_stop_km', 'dist_to_university_km',
       'Population', 'Άγαμοι_pct', 'Έγγαμοι_pct', 'Χήροι_pct',
       'Διαζευγμένοι_pct', 'age_0_14_pct', 'age_15_64_pct', 'age_65_plus_pct',
       'low_education_pct', 'medium_education_pct', 'high_education_pct',
       'unemployment_rate', 'labor_force_participation_rate',
       'primary_sector_pct', 'secondary_sector_pct', 'tertiary_sector_pct',
       'Area_km2', 'Rating'],
      dtype='object')


,Latitude,Longitude,NACE Code,Neighborhood_Area_km2,distance_to_volos_center_km,distance_to_volos_port_km,dist_to_main_road_km,dist_to_bus_stop_km,dist_to_university_km,Population,...,low_education_pct,medium_education_pct,high_education_pct,unemployment_rate,labor_force_participation_rate,primary_sector_pct,secondary_sector_pct,tertiary_sector_pct,Area_km2,Rating
0,39.335293,22.923506,47.76,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331,85806,...,0.3634,0.3663,0.2704,0.1545,0.4188,0.0313,0.1666,0.8022,26.790807,NaN
1,39.339233,22.923969,47.59,4.848904,3.508166,2.529372,0.224486,0.246542,1.913456,85806,...,0.3634,0.3663,0.2704,0.1545,0.4188,0.0313,0.1666,0.8022,26.790807,NaN
2,39.332305,22.926496,94.91,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331,85806,...,0.3634,0.3663,0.2704,0.1545,0.4188,0.0313,0.1666,0.8022,26.790807,NaN
3,39.332756,22.929457,49.41,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331,85806,...,0.3634,0.3663,0.2704,0.1545,0.4188,0.0313,0.1666,0.8022,26.790807,NaN
4,39.332876,22.929374,47.11,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331,85806,...,0.3634,0.3663,0.2704,0.1545,0.4188,0.0313,0.1666,0.8022,26.790807,NaN


### ***Divide Labeled vs Unlabeled Entries***

In [21]:
# === 2. Preprocessing ===

# Separate labeled vs unlabeled
df_labeled = df[df['Rating'].notna()].copy()
df_unlabeled = df[df['Rating'].isna()].copy()

print(f"Number of labeled samples: {len(df_labeled)}")
print(f"Number of unlabeled samples: {len(df_unlabeled)}")

Number of labeled samples: 228
Number of unlabeled samples: 3654


In [22]:
X_labeled = df_labeled.drop(columns='Rating')
y_labeled = df_labeled['Rating']
X_unlabeled = df_unlabeled.drop(columns='Rating')

print(f"{len(X_labeled.columns)} colunms in X_labeled: {X_labeled.columns}\n")
print(f"Target column: y_labeled, dtype: {y_labeled.dtype}\n")
print(f"{len(X_unlabeled.columns)} colunms in X_unlabeled: {X_unlabeled.columns}\n")

26 colunms in X_labeled: Index(['Latitude', 'Longitude', 'NACE Code', 'Neighborhood_Area_km2',
       'distance_to_volos_center_km', 'distance_to_volos_port_km',
       'dist_to_main_road_km', 'dist_to_bus_stop_km', 'dist_to_university_km',
       'Population', 'Άγαμοι_pct', 'Έγγαμοι_pct', 'Χήροι_pct',
       'Διαζευγμένοι_pct', 'age_0_14_pct', 'age_15_64_pct', 'age_65_plus_pct',
       'low_education_pct', 'medium_education_pct', 'high_education_pct',
       'unemployment_rate', 'labor_force_participation_rate',
       'primary_sector_pct', 'secondary_sector_pct', 'tertiary_sector_pct',
       'Area_km2'],
      dtype='object')

Target column: y_labeled, dtype: float64

26 colunms in X_unlabeled: Index(['Latitude', 'Longitude', 'NACE Code', 'Neighborhood_Area_km2',
       'distance_to_volos_center_km', 'distance_to_volos_port_km',
       'dist_to_main_road_km', 'dist_to_bus_stop_km', 'dist_to_university_km',
       'Population', 'Άγαμοι_pct', 'Έγγαμοι_pct', 'Χήροι_pct',
       'Διαζευ

### ***Encode NACE Code as Categorical Feature***

In [23]:
# Identify numerical and categorical columns
categorical_cols = ['NACE Code']
numerical_cols = [col for col in selected_features if col != 'NACE Code' and col != 'Rating']
print(f"{len(categorical_cols)} categorical columns: {categorical_cols}")
print(f"{len(numerical_cols)} numerical columns: {numerical_cols}")

1 categorical columns: ['NACE Code']
25 numerical columns: ['Latitude', 'Longitude', 'Neighborhood_Area_km2', 'distance_to_volos_center_km', 'distance_to_volos_port_km', 'dist_to_main_road_km', 'dist_to_bus_stop_km', 'dist_to_university_km', 'Population', 'Άγαμοι_pct', 'Έγγαμοι_pct', 'Χήροι_pct', 'Διαζευγμένοι_pct', 'age_0_14_pct', 'age_15_64_pct', 'age_65_plus_pct', 'low_education_pct', 'medium_education_pct', 'high_education_pct', 'unemployment_rate', 'labor_force_participation_rate', 'primary_sector_pct', 'secondary_sector_pct', 'tertiary_sector_pct', 'Area_km2']


In [24]:
# Preprocess data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ]), numerical_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ])

In [25]:
# Apply preprocessing
X_labeled_processed = preprocessor.fit_transform(X_labeled)
X_unlabeled_processed = preprocessor.transform(X_unlabeled)

### ***Perform Self-Training***

In [26]:
# Initialize regressor
base_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

In [27]:
# Self-training loop
confidence_threshold = 0.1  # threshold for high-confidence predictions
max_iterations = 10

X_train = X_labeled_processed
y_train = y_labeled
X_unlabeled_pool = X_unlabeled_processed.copy()


for iteration in range(max_iterations):

    # Train the model
    base_regressor.fit(X_train, y_train)

    # Predict the unlabeled data target
    predictions = base_regressor.predict(X_unlabeled_pool)

    # Find the confident predictions
    tree_predictions = np.array([tree.predict(X_unlabeled_pool) for tree in base_regressor.estimators_])
    variances = np.var(tree_predictions, axis=0)
    high_confidence_idx = variances < confidence_threshold
    
    if not high_confidence_idx.any():
        print(f"No high-confidence predictions in iteration {iteration + 1}. Stopping.")
        break


    # Add the confident predictions to the labeled set
    X_new_labeled = X_unlabeled_pool[high_confidence_idx]
    y_new_labeled = predictions[high_confidence_idx]
    
    # Update the training set
    X_train = np.vstack((X_train, X_new_labeled))
    y_train = np.hstack((y_train, y_new_labeled))
    
    # Remove the confident predictions from the unlabeled set
    X_unlabeled_pool = X_unlabeled_pool[~high_confidence_idx]
    print(f"Iteration {iteration + 1}: Added {len(X_new_labeled)} new labeled samples")
    
    if len(X_unlabeled_pool) == 0:
        print("All unlabeled data labeled.")
        break
    

Iteration 1: Added 1046 new labeled samples
Iteration 2: Added 570 new labeled samples
Iteration 3: Added 709 new labeled samples
Iteration 4: Added 37 new labeled samples
Iteration 5: Added 1 new labeled samples
Iteration 6: Added 1 new labeled samples
No high-confidence predictions in iteration 7. Stopping.


In [28]:
# Infrerence on the unlabeled data
final_predictions = base_regressor.predict(X_unlabeled_processed)

# Clip predictions to ensure they are within 1–5
final_predictions = np.clip(final_predictions, 1, 5)

# Update the original dataset
df.loc[df['Rating'].isnull(), 'Rating'] = final_predictions

In [29]:
# Assign the new ratings to the original DataFrame

original_df = pd.read_csvdf = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\4. Joining Datasets\\3. Joining - Business - Neighborhood Data\\Extracted CSV Files\\business_data_ELSTAT_neighborhood.csv")

columns_to_compare = [col for col in df.columns if col != 'Rating']
matching_rows = (original_df[columns_to_compare] == df[columns_to_compare]).all(axis=1)
if not matching_rows.all():
    print("⚠️ Warning: Some rows do not match between the original and new DataFrames.")
    mismatches = (~matching_rows).sum()
    print(f"Non-matching rows: {mismatches}")


# Only update where original Rating is missing AND rows match
original_df.loc[original_df['Rating'].isna() & matching_rows, 'Rating'] = \
    df.loc[original_df['Rating'].isna() & matching_rows, 'Rating']

### ***Save the Data***

In [30]:
print(original_df.shape)
print(original_df.columns)
print(f"Missing ratings: {original_df['Rating'].isna().sum()}")
original_df.head()

(3882, 42)
Index(['Name', 'Category', 'Latitude', 'Longitude', 'Address', 'Country',
       'City', 'Postal Code', 'Rating', 'Reviews', 'Source', 'Description',
       'NACE Code', 'NACE Description (EN)', 'Municipal_Community',
       'Neighborhood', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ', 'ΔΗΜΟΣ', 'ΔΗΜΟΤΙΚΗ ΕΝΟΤΗΤΑ',
       'Population', 'Άγαμοι_pct', 'Έγγαμοι_pct', 'Χήροι_pct',
       'Διαζευγμένοι_pct', 'age_0_14_pct', 'age_15_64_pct', 'age_65_plus_pct',
       'low_education_pct', 'medium_education_pct', 'high_education_pct',
       'unemployment_rate', 'labor_force_participation_rate',
       'primary_sector_pct', 'secondary_sector_pct', 'tertiary_sector_pct',
       'Area_km2', 'Neighborhood_Area_km2', 'distance_to_volos_center_km',
       'distance_to_volos_port_km', 'dist_to_main_road_km',
       'dist_to_bus_stop_km', 'dist_to_university_km'],
      dtype='object')
Missing ratings: 0


,Name,Category,Latitude,Longitude,Address,Country,City,Postal Code,Rating,Reviews,...,primary_sector_pct,secondary_sector_pct,tertiary_sector_pct,Area_km2,Neighborhood_Area_km2,distance_to_volos_center_km,distance_to_volos_port_km,dist_to_main_road_km,dist_to_bus_stop_km,dist_to_university_km
0,Άνθη-φυτά,Flower Store,39.335293,22.923506,6χλ.βολου,Greece,Βόλος Μαγνησίας,38334,3.040418,NaN,...,0.0313,0.1666,0.8022,26.790807,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331
1,Μέταλλο και ξύλο,Furniture and Home Store,39.339233,22.923969,Βόλου - Αθηνών 7ο χλμ,Greece,Βόλος Μαγνησίας,38334,4.952722,NaN,...,0.0313,0.1666,0.8022,26.790807,4.848904,3.508166,2.529372,0.224486,0.246542,1.913456
2,Προφήτης Ηλίας Αλυκών,Church,39.332305,22.926496,Λεωφόρος Αθηνών 155,Greece,Βόλος Μαγνησίας,38334,3.039758,NaN,...,0.0313,0.1666,0.8022,26.790807,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331
3,Frago Cargo (Φραγγοσ Νικολαοσ),"Shipping, Freight, and Material Transportation...",39.332756,22.929457,Αλόης 179Γ,Greece,Βόλος Μαγνησίας,38334,3.079758,NaN,...,0.0313,0.1666,0.8022,26.790807,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331
4,Αλφα Ωμεγα Express Market,Grocery Store,39.332876,22.929374,Βάκχου 4,Greece,Βόλος Μαγνησίας,38334,3.107334,NaN,...,0.0313,0.1666,0.8022,26.790807,5.438003,5.302196,4.293831,0.404907,0.534738,3.883331


In [31]:
original_df.to_csv('C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\6. Ground Truths\\3. Semi-Supervised Approach\\2. Approach 2\\Extracted CSV Files\\business_data_imputed.csv', index=False)